# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)   # START

        # Estanterías: el robot no puede entrar
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (celdas amarillas)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Terminales: el episodio termina al llegar
        self.terminal_states = {
            (0, 5): 10.0,    # ENTREGA +10
            (2, 2): 2.0,     # CARGA +2
            (3, 5): -10.0,   # peligro mortal -10
        }

        # Peligros NO terminales: cuestan -3 pero el robot sigue
        self.danger_states = {(1, 4): -3.0, (4, 1): -3.0}

        self.living_reward = -1.0
        self.gamma = 0.9

        # NUEVO: probabilidades (dirección elegida, cada desviación)
        self.p_normal = (0.90, 0.05)
        self.p_slippery = (0.60, 0.20)

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False                      # fuera del grid
        return state not in self.walls        # no es estantería

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if self.is_valid_state((r, c))]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]   # +10, +2 o -10
        if state in self.danger_states:
            return self.danger_states[state]     # -3, no termina
        return self.living_reward                # costo por paso

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]
        """
        if self.is_terminal(state):
            return [(state, 1.0)]   # terminal absorbente

        # NUEVO: se leen de los atributos según el piso del estado ACTUAL
        if state in self.slippery_states:
            p_intended, p_side = self.p_slippery
        else:
            p_intended, p_side = self.p_normal

        # Desviaciones perpendiculares a la acción
        perp1 = ( action[1],  action[0])
        perp2 = (-action[1], -action[0])

        transitions = []
        for move, prob in [(action, p_intended), (perp1, p_side), (perp2, p_side)]:
            next_state = (state[0] + move[0], state[1] + move[1])
            if not self.is_valid_state(next_state):
                next_state = state   # choca: se queda quieto
            transitions.append((next_state, prob))
        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # Σ T(s,a,s') V(s'): valor esperado del siguiente estado,
    # ponderando cada destino posible por su probabilidad
    return sum(p * V[s2] for s2, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}   # V_0(s) = 0

    for it in range(max_iter):
        V_new = {}   # dict nuevo: todos se actualizan con los valores de V_k
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)   # terminal: no hay futuro, V = R(s)
            else:
                # Bellman: R(s) + γ · max_a Σ T(s,a,s') V_k(s')
                V_new[s] = grid.get_reward(s) + grid.gamma * max(
                    expected_next_value(grid, s, a, V) for a in grid.actions)

        # Convergencia: el estado que más cambió, cambió menos que θ
        delta = max(abs(V_new[s] - V[s]) for s in V)
        V = V_new
        if delta < threshold:
            break

    return V, it + 1


def extract_policy(grid, V):
    # π*(s) = argmax_a Σ T(s,a,s') V*(s')
    # R(s) y γ no se incluyen: son iguales para todas las acciones de s
    return {
        s: max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
        for s in grid.states() if not grid.is_terminal(s)
    }


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # Calcula V^π: cuánto vale cada estado si SIEMPRE sigo la política dada
    V = {s: 0.0 for s in grid.states()}

    for _ in range(max_iter):
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                # R(s) + γ · Σ T(s,π(s),s') V(s')  → SIN max: la acción ya está fijada por π
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(
                    grid, s, policy[s], V)

        delta = max(abs(V_new[s] - V[s]) for s in V)
        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # π_new(s) = argmax_a Σ T(s,a,s') V^π(s')
    # Es la misma operación que extract_policy, así que la reutilizamos
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. Política inicial arbitraria: todos los estados van hacia arriba
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []   # cuántos estados cambiaron de acción en cada vuelta

    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)   # 2. evaluar
        new_policy = policy_improvement(grid, V)          # 3. mejorar

        changes = sum(new_policy[s] != policy[s] for s in policy)
        history.append(changes)
        policy = new_policy

        if changes == 0:   # 4. estable: ningún estado cambió → política óptima
            break

    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

Prefiere la estación de carga +2. Desde START la política sigue → → ↓ ↓:
(0,0) → (0,1) → (0,2) → (1,2) → (2,2). V(START) = -2.575.

2. ¿Por qué una recompensa menor podría ser óptima?

Porque cada paso cuesta -1 y el futuro se descuenta con γ = 0.9. La entrega +10
está a 7 pasos, pasando junto a un peligro -3. La carga +2 está a 4 pasos. Al restar el costo acumulado y el descuento, llegar rápido a +2 vale más que el camino largo a +10.

3. ¿En qué estados el piso resbaloso cambia la decisión?

En (1,2), que es resbaloso: desde ahí el robot se desvía con 40 % de probabilidad.
En lugar de seguir → hacia la entrega, prefiere ↓ hacia la carga, que está al lado.
Si quitamos el piso resbaloso, en (1,2) la política cambia a → y busca el +10.
También cambia la decisión en (3,1), junto a la celda resbalosa (2,1).

4. ¿Qué papel cumple el costo por paso `-1`?

Es un castigo por tardar: empuja al robot a terminar el episodio rápido.
Cuanto más negativo es, más atractivas se vuelven las metas cercanas aunque den menos recompensa.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Porque la probabilidad de desviarse depende del tipo de piso del estado actual:
90/5/5 en piso normal y 60/20/20 en piso resbaloso. T(s,a,s') depende de s,
no solo de la acción.


### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [7]:
def correr_experimento(nombre, grid):
    V, n = value_iteration(grid)
    pi = extract_policy(grid, V)
    print(f"=== {nombre} ===  (iteraciones: {n}, V(START) = {V[grid.start]:+.3f})")
    print_policy(grid, pi)
    print()

# A — Menos costo por paso
grid_A = WarehouseMDP()
grid_A.living_reward = -0.1
correr_experimento("A: living_reward = -0.1", grid_A)

# B — Piso muy resbaloso: 0.40 deseado, 0.30 a cada lado
grid_B = WarehouseMDP()
grid_B.p_slippery = (0.40, 0.30)
correr_experimento("B: piso resbaloso 0.40", grid_B)

# C — Más paciencia
grid_C = WarehouseMDP()
grid_C.gamma = 0.99
correr_experimento("C: gamma = 0.99", grid_C)

# Bonus — buscar el living_reward donde START cambia de destino
for lr in [-0.70, -0.75, -0.79, -0.80, -0.85, -0.90]:
    g = WarehouseMDP()
    g.living_reward = lr
    pi = extract_policy(g, value_iteration(g)[0])
    destino = "+10 (entrega)" if pi[(1, 2)] == (0, 1) else "+2 (carga)"
    print(f"living_reward = {lr:+.2f}  →  {destino}")

=== A: living_reward = -0.1 ===  (iteraciones: 26, V(START) = +1.649)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== B: piso resbaloso 0.40 ===  (iteraciones: 22, V(START) = -2.706)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== C: gamma = 0.99 ===  (iteraciones: 24, V(START) = -1.456)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

living_reward = -0.70  →  +10 (entrega)
living_reward = -0.75  →  +10 (entrega)
living_reward = -0.79  →  +10 (entrega)
living_reward = -0.80  →  +2 (carga)
living_reward = -0.85  →  +2 (carga)
living_reward = -0.90  →  +2 (carga)


### Resultados

**A (living_reward = -0.1):** predije que iría al +10. Sí: desde START ahora va a la entrega +10.

**B (resbaloso 0.40):** la política no cambia, sigue yendo a la carga +2. Solo baja V(START) (de -2.575 a -2.706).

**C (gamma = 0.99):** sí, valora más el +10 distante. Desde START ahora va a la entrega.

**Bonus:** el cambio ocurre entre living_reward = -0.79 (va a +10) y -0.80 (va a +2).